# 🔍 Phase 1: Data Preparation, Missing Value Imputation & EDA per Project

**Penelitian**: *Predicting Flaky Test in Cross-Project Scenario*

Dokumen notebook ini memproses **Phase 1** dari roadmap penelitian flaky test:
1. **Load Data**: Membaca dataset `data/test_features.csv`.
2. **Pemisahan Metadata vs Matriks Fitur ($X$)**:
   - `project`: Disimpan sebagai variabel stratifikasi / *grouping* untuk *Leave-One-Project-Out Cross Validation (LOPO-CV)*.
   - `test_name`: Disimpan sebagai *Primary Key* pelacakan test case.
   - Redundansi identitas (`Unnamed: 0`, `testClassName`, `testMethodName`) dibuang dari matriks fitur $X$.
3. **Analisis & Imputasi Missing Values**:
   - Pengecekan data hilang pada `testLength`, `numAsserts`, dan `numCoveredLines`.
   - Imputasi probabilitas terkelompok (**Group-Aware Probabilistic & Per-Project Median Imputation**) untuk menjaga integritas skala per proyek.
4. **Exploratory Data Analysis (EDA) per Proyek**:
   - Statistik deskriptif dan ketimpangan kelas target (`flaky`) antar proyek.
   - Analisis *domain shift* dan perbedaan skala fitur numerik.
   - Pengecekan multikolinearitas antar fitur prediktif.
5. **Penyimpanan Dataset Bersih**: Ke `data/cleaned_test_features.csv`.

## 1. Import Library & Load Raw Dataset

In [1]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Konfigurasi visualisasi
sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams["figure.figsize"] = (12, 6)
plt.rcParams["font.size"] = 10

# Path dataset
data_path = "../data/test_features.csv"
df_raw = pd.read_csv(data_path)

print(f"Dataset Raw Shape: {df_raw.shape[0]} baris, {df_raw.shape[1]} kolom")
print("Kolom-kolom Dataset:")
print(df_raw.columns.tolist())
df_raw.head(3)

Dataset Raw Shape: 22236 baris, 29 kolom
Kolom-kolom Dataset:
['Unnamed: 0', 'test_name', 'project', 'testClassName', 'testMethodName', 'flaky', 'assertion-roulette', 'conditional-test-logic', 'eager-test', 'fire-and-forget', 'indirect-testing', 'mystery-guest', 'resource-optimism', 'test-run-war', 'testLength', 'numAsserts', 'numCoveredLines', 'ExecutionTime', 'projectSourceLinesCovered', 'projectSourceClassesCovered', 'hIndexModificationsPerCoveredLine_window5', 'hIndexModificationsPerCoveredLine_window10', 'hIndexModificationsPerCoveredLine_window25', 'hIndexModificationsPerCoveredLine_window50', 'hIndexModificationsPerCoveredLine_window75', 'hIndexModificationsPerCoveredLine_window100', 'hIndexModificationsPerCoveredLine_window500', 'hIndexModificationsPerCoveredLine_window10000', 'num_third_party_libs']


## 2. Pemisahan Metadata, Primary Key, dan Matriks Fitur ($X$)

In [2]:
# 1. Kolom Identifikasi & Stratifikasi Evaluasi (Metadata)
metadata_cols = ['project', 'test_name']

# 2. Kolom Identitas Berlebih yang Dibuang
dropped_cols = ['Unnamed: 0', 'testClassName', 'testMethodName']

# 3. Target Variable
target_col = 'flaky'

# 4. Matriks Fitur Prediktif X (22 Fitur)
feature_cols = [c for c in df_raw.columns if c not in metadata_cols + dropped_cols + [target_col]]

test_smells = [c for c in feature_cols if c in [
    'assertion-roulette', 'conditional-test-logic', 'eager-test', 'fire-and-forget',
    'indirect-testing', 'mystery-guest', 'resource-optimism', 'test-run-war'
]]
test_metrics = ['testLength', 'numAsserts', 'numCoveredLines', 'ExecutionTime']
coverage_features = ['projectSourceLinesCovered', 'projectSourceClassesCovered']
churn_features = [c for c in feature_cols if 'hIndex' in c]
dep_features = ['num_third_party_libs']

print(f"Total Fitur Prediktif dalam Matriks X: {len(feature_cols)} fitur")
print("\nPengelompokan Fitur Prediktif:")
print(f" - Test Smells ({len(test_smells)}): {test_smells}")
print(f" - Test Metrics ({len(test_metrics)}): {test_metrics}")
print(f" - Coverage Features ({len(coverage_features)}): {coverage_features}")
print(f" - Code Churn Features ({len(churn_features)}): {churn_features}")
print(f" - Dependency ({len(dep_features)}): {dep_features}")

Total Fitur Prediktif dalam Matriks X: 23 fitur

Pengelompokan Fitur Prediktif:
 - Test Smells (8): ['assertion-roulette', 'conditional-test-logic', 'eager-test', 'fire-and-forget', 'indirect-testing', 'mystery-guest', 'resource-optimism', 'test-run-war']
 - Test Metrics (4): ['testLength', 'numAsserts', 'numCoveredLines', 'ExecutionTime']
 - Coverage Features (2): ['projectSourceLinesCovered', 'projectSourceClassesCovered']
 - Code Churn Features (8): ['hIndexModificationsPerCoveredLine_window5', 'hIndexModificationsPerCoveredLine_window10', 'hIndexModificationsPerCoveredLine_window25', 'hIndexModificationsPerCoveredLine_window50', 'hIndexModificationsPerCoveredLine_window75', 'hIndexModificationsPerCoveredLine_window100', 'hIndexModificationsPerCoveredLine_window500', 'hIndexModificationsPerCoveredLine_window10000']
 - Dependency (1): ['num_third_party_libs']


## 3. Analisis Missing Values & Strategy Imputasi Probabilitas / Per-Project

Pemeriksaan missing values pada seluruh fitur prediktif.

In [3]:
# Ringkasan Missing Value
null_counts = df_raw[feature_cols].isnull().sum()
null_summary = null_counts[null_counts > 0]
print("=== Ringkasan Missing Values Raw Data ===")
print(null_summary)

# Detail sebaran per proyek
missing_rows = df_raw[df_raw['testLength'].isnull()]
print("\nMissing values per proyek asal:")
print(missing_rows['project'].value_counts())
print("\nMissing values per status target (flaky):")
print(missing_rows['flaky'].value_counts())

=== Ringkasan Missing Values Raw Data ===
testLength         273
numAsserts         273
numCoveredLines    273

Missing values per proyek asal:
project
wildfly            206
logback             38
spring-boot         12
wro4j               10
handlebars.java      7

Missing values per status target (flaky):
flaky
0    270
1      3


### Implementasi Imputasi Terkelompok per Proyek (Group-Aware Probabilistic Imputation)

Strategi imputasi yang digunakan:
1. **Primary Strategy**: Mengisi nilai NaN dengan **Median dari Proyek yang sama**. Ini memastikan skala unik proyek (misal `wildfly` vs `handlebars.java`) tetap terjaga.
2. **Fallback Probabilistic Imputation**: Jika suatu proyek memiliki seluruh nilai NaN pada kolom tersebut, lakukan sampling acak dari distribusi empiris non-null fitur tersebut.

In [4]:
def impute_project_aware(df, cols_to_impute):
    df_imputed = df.copy()
    np.random.seed(42)
    
    for col in cols_to_impute:
        # 1. Imputasi dengan Median per Proyek
        df_imputed[col] = df_imputed.groupby('project')[col].transform(lambda x: x.fillna(x.median()))
        
        # 2. Fallback Probabilistic Sampling jika masih ada NaN
        if df_imputed[col].isnull().sum() > 0:
            non_null_vals = df_imputed[col].dropna().values
            nan_indices = df_imputed[df_imputed[col].isnull()].index
            sampled_vals = np.random.choice(non_null_vals, size=len(nan_indices), replace=True)
            df_imputed.loc[nan_indices, col] = sampled_vals
            
    return df_imputed

cols_missing = null_summary.index.tolist()
df_clean = impute_project_aware(df_raw, cols_missing)

print("=== Memulai Imputasi Project-Aware ===")
print(f"Fitur yang di-imputasi: {cols_missing}")
print("Imputasi selesai!")
print(f"Total Missing Values setelah Imputasi: {df_clean[feature_cols].isnull().sum().sum()}")

=== Memulai Imputasi Project-Aware ===
Fitur yang di-imputasi: ['testLength', 'numAsserts', 'numCoveredLines']
Imputasi selesai!
Total Missing Values setelah Imputasi: 0


## 4. Exploratory Data Analysis (EDA) per Proyek

### 4.1 Distribusi Class Target (`flaky`) per Proyek

In [5]:
project_stats = df_clean.groupby('project')['flaky'].agg(
    total_tests='count',
    flaky_count='sum',
    flaky_ratio=lambda x: x.mean() * 100
).reset_index().sort_values(by='flaky_ratio', ascending=False)

plt.figure(figsize=(14, 6))
ax = sns.barplot(data=project_stats, x='project', y='flaky_ratio', palette='viridis')
plt.xticks(rotation=60, ha='right')
plt.ylabel('Rasio Flaky Test (%)', fontsize=12)
plt.xlabel('Proyek Target', fontsize=12)
plt.title('Rasio Flaky Test per Proyek (24 Open-Source Projects)', fontsize=14, fontweight='bold')
for p in ax.patches:
    if p.get_height() > 0:
        ax.annotate(f"{p.get_height():.1f}%", (p.get_x() + p.get_width() / 2., p.get_height()),
                    ha='center', va='bottom', fontsize=8, rotation=0, xytext=(0, 2), textcoords='offset points')
plt.tight_layout()
plt.show()

print(project_stats.to_string(index=False))

### 4.2 Inspeksi Skala & Domain Shift Fitur Numerik Lintas Proyek

Menganalisis perbedaan skala antar proyek untuk fitur `ExecutionTime`, `numCoveredLines`, `projectSourceLinesCovered`, dan `testLength`.

In [6]:
fig, axes = plt.subplots(2, 2, figsize=(16, 10))

sns.boxplot(data=df_clean, x='project', y='ExecutionTime', ax=axes[0,0])
axes[0,0].set_yscale('log')
axes[0,0].set_title('ExecutionTime (Detik - Skala Log)', fontweight='bold')
axes[0,0].tick_params(axis='x', rotation=90)

sns.boxplot(data=df_clean, x='project', y='numCoveredLines', ax=axes[0,1])
axes[0,1].set_yscale('log')
axes[0,1].set_title('numCoveredLines (Skala Log)', fontweight='bold')
axes[0,1].tick_params(axis='x', rotation=90)

sns.boxplot(data=df_clean, x='project', y='projectSourceLinesCovered', ax=axes[1,0])
axes[1,0].set_yscale('log')
axes[1,0].set_title('projectSourceLinesCovered (Skala Log)', fontweight='bold')
axes[1,0].tick_params(axis='x', rotation=90)

sns.boxplot(data=df_clean, x='project', y='testLength', ax=axes[1,1])
axes[1,1].set_yscale('log')
axes[1,1].set_title('testLength (LOC Test - Skala Log)', fontweight='bold')
axes[1,1].tick_params(axis='x', rotation=90)

plt.tight_layout()
plt.show()

### 4.3 Analisis Multikolinearitas antar Fitur Prediktif

Mengukur korelasi Spearman (robust terhadap korelasi non-linier & outlier) antar 22 fitur prediktif.

In [7]:
plt.figure(figsize=(14, 10))
corr = df_clean[feature_cols].corr(method='spearman')
sns.heatmap(corr, cmap='coolwarm', vmin=-1, vmax=1, annot=False, linewidths=0.5)
plt.title('Heatmap Korelasi Spearman 22 Fitur Prediktif', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

## 5. Simpan Dataset Hasil Cleaning Phase 1

In [8]:
output_path = "../data/cleaned_test_features.csv"
df_clean.to_csv(output_path, index=False)
print(f"✅ Dataset bersih Phase 1 berhasil disimpan ke: {output_path}")
print(f"Ukuran dataset: {df_clean.shape[0]} baris, {df_clean.shape[1]} kolom")

✅ Dataset bersih Phase 1 berhasil disimpan ke: ../data/cleaned_test_features.csv
Ukuran dataset: 22236 baris, 29 kolom
